# Level 2: distill Depth Anything V2 into a small CNN

**Teacher:** Depth Anything V2 Small (`vits`, about 25M parameters), the model from Level 1.
**Student:** the U-Net from `segment_pets_at_oxford.ipynb` (about 7.8M parameters, 1 output channel), trained to copy the teacher's depth maps.

How it works:

1. Run the teacher once over a set of photos and save its depth map for each one. These are the student's training targets ("teacher labels"), so no human labelling is needed.
2. Train the student to reproduce those depth maps with **L1 loss** (mean absolute difference per pixel). Unlike the pets notebook there is no sigmoid or threshold: the output is a continuous depth value, not a yes/no mask.
3. Plug the student into the Level 1 webcam script in place of the teacher. The NEAR/FAR markers and grid work unchanged.

Expect the student to be blurrier than the teacher: it is smaller, sees 160x120 images, and (by default) has only seen pet photos. Step 2 lets you add your own webcam frames, which is the first thing to try if the webcam results look poor.

## Step 1: Setup

Install `kagglehub` (downloads the Oxford-IIIT Pet photos). Run this once.

In [ ]:
%pip install kagglehub

Move into the repo folder (so `depth_anything_v2` imports and files land in the git-ignored folder) and set paths.

In [ ]:
%cd C:\Users\afeta\eye-in-the-sky\Depth-Anything-V2

import glob
import importlib
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from depth_anything_v2.dpt import DepthAnythingV2

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
DATA_DIR = 'level2_data'                          # teacher labels and webcam frames (inside the ignored folder)
WEBCAM_DIR = os.path.join(DATA_DIR, 'webcam')
LABELS_PATH = os.path.join(DATA_DIR, 'teacher_labels.npz')
STUDENT_PATH = 'checkpoints/student_unet.pth'
os.makedirs(WEBCAM_DIR, exist_ok=True)
print('Using device:', DEVICE)

### The student model

This cell writes `student_unet.py` so the webcam script can import the same model. The U-Net is copied from the pets notebook unchanged.

`StudentDepth.infer_image(frame)` takes a BGR webcam frame and returns a depth map at the frame's size, the same call the webcam script already makes on the teacher. Training and the webcam both go through `resize_rgb` and `to_input`, so the student always sees images prepared the same way.

`IMG_H, IMG_W = 120, 160` keeps the webcam's 4:3 shape. Both must be multiples of 8 because the U-Net halves the size three times.

In [ ]:
%%writefile student_unet.py
"""Level 2 student: the U-Net from segment_pets_at_oxford.ipynb, predicting one depth channel."""
import cv2
import torch
import torch.nn as nn

IMG_H, IMG_W = 120, 160  # 4:3 like the webcam. Both must be multiples of 8 (three 2x poolings).
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)

        # Decoder with Skip Connections
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Final Conv 1x1
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        p1 = self.pool1(s1)

        s2 = self.enc2(p1)
        p2 = self.pool2(s2)

        s3 = self.enc3(p2)
        p3 = self.pool3(s3)

        b = self.bottleneck(p3)

        d3 = self.up3(b)
        d3 = torch.cat([d3, s3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, s2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, s1], dim=1)
        d1 = self.dec1(d1)

        return self.final_conv(d1)


def resize_rgb(bgr):
    """BGR image of any size -> RGB uint8 at IMG_W x IMG_H, exactly what the student sees."""
    small = cv2.resize(bgr, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(small, cv2.COLOR_BGR2RGB)


def to_input(rgb_uint8, device):
    """N x H x W x 3 uint8 batch (numpy or tensor) -> normalized N x 3 x H x W float tensor on device."""
    x = torch.as_tensor(rgb_uint8).to(device).permute(0, 3, 1, 2).float() / 255.0
    return (x - MEAN.to(device)) / STD.to(device)


class StudentDepth:
    """A trained UNet behind the same infer_image() call as DepthAnythingV2."""

    def __init__(self, checkpoint, device):
        self.device = device
        self.net = UNet(3, 1)
        self.net.load_state_dict(torch.load(checkpoint, map_location='cpu'))
        self.net = self.net.to(device).eval()

    @torch.no_grad()
    def infer_image(self, frame, input_size=None):
        # input_size is accepted for compatibility and ignored: the student always runs at IMG_W x IMG_H.
        depth = self.net(to_input(resize_rgb(frame)[None], self.device))[0, 0].float().cpu().numpy()
        return cv2.resize(depth, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_LINEAR)

In [ ]:
import student_unet
importlib.reload(student_unet)  # pick up edits if the cell above was re-run
from student_unet import IMG_H, IMG_W, UNet, resize_rgb, to_input

print('Student parameters:', sum(p.numel() for p in UNet(3, 1).parameters()))

## Step 2 (optional): Add your own webcam frames

The pet photos are mostly close-ups of animals, which look nothing like a room seen from a webcam. Mixing in a few hundred of your own frames usually helps the student a lot on the webcam.

Set `CAPTURE_WEBCAM = True` and run the cell. A preview window opens and saves a frame every `CAPTURE_EVERY` seconds. Move the camera around: point it at walls, doorways, furniture, your hand close up, people at different distances. Press **q** to stop early. Running it again adds more frames. Then re-run Step 3, which notices the new frames and rebuilds the labels.

In [ ]:
CAPTURE_WEBCAM = False  # set True to record frames
CAPTURE_SECONDS = 60
CAPTURE_EVERY = 0.2     # seconds between saved frames (60 s -> about 300 frames)
CAMERA_INDEX = 0

if CAPTURE_WEBCAM:
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW) if os.name == 'nt' else cv2.VideoCapture(CAMERA_INDEX)
    count = len(glob.glob(os.path.join(WEBCAM_DIR, '*.jpg')))
    start = last_save = time.time()
    while cap.isOpened() and time.time() - start < CAPTURE_SECONDS:
        ok, frame = cap.read()
        if not ok:
            break
        if time.time() - last_save >= CAPTURE_EVERY:
            cv2.imwrite(os.path.join(WEBCAM_DIR, f'{count:05d}.jpg'), frame)
            count += 1
            last_save = time.time()
        preview = frame.copy()
        cv2.putText(preview, f'saved {count}  (q to stop)', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow('Capturing training frames', preview)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

print('Webcam frames on disk:', len(glob.glob(os.path.join(WEBCAM_DIR, '*.jpg'))))

## Step 3: Make the teacher labels

Download the pet photos (about 800 MB, cached after the first run), then run Depth Anything V2 over every image. Each depth map is shrunk to the student's size and rescaled to 0-1 (1 = nearest point in that image), and saved together with the shrunken image in `level2_data/teacher_labels.npz`.

This takes a few minutes on the GPU the first time. Later runs load the saved file instantly, unless the set of images changed (for example after capturing webcam frames).

In [ ]:
import kagglehub

pets_path = kagglehub.dataset_download('tanlikesmath/the-oxfordiiit-pet-dataset')
# The download holds every photo twice (images/ and an identical images/images/), so keep one file per name.
pet_files = sorted({os.path.basename(f): f for f in glob.glob(os.path.join(pets_path, '**', '*.jpg'), recursive=True)}.values())
webcam_files = sorted(glob.glob(os.path.join(WEBCAM_DIR, '*.jpg')))
files = pet_files + webcam_files
print(f'{len(pet_files)} pet photos + {len(webcam_files)} webcam frames')

In [ ]:
TEACHER_INPUT_SIZE = 518  # labels are made once, so use the repo's full-quality setting (Level 1 uses 308 for speed)

cached = np.load(LABELS_PATH) if os.path.exists(LABELS_PATH) else None
if cached is not None and list(cached['files']) == files:
    images, depths, is_webcam = cached['images'], cached['depths'], cached['is_webcam']
    print('Loaded saved teacher labels.')
else:
    teacher = DepthAnythingV2(encoder='vits', features=64, out_channels=[48, 96, 192, 384])
    teacher.load_state_dict(torch.load('checkpoints/depth_anything_v2_vits.pth', map_location='cpu'))
    teacher = teacher.to(DEVICE).eval()

    images, depths, kept = [], [], []
    t0 = time.time()
    for i, f in enumerate(files):
        bgr = cv2.imread(f)
        if bgr is None:  # a few files in the pet dataset are not readable JPEGs
            continue
        d = teacher.infer_image(bgr, TEACHER_INPUT_SIZE)
        d = cv2.resize(d, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
        d = (d - d.min()) / (d.max() - d.min() + 1e-8)  # 0 = farthest, 1 = nearest in this image
        images.append(resize_rgb(bgr))
        depths.append(d.astype(np.float16))
        kept.append(f)
        if (i + 1) % 500 == 0:
            print(f'{i + 1}/{len(files)} images, {time.time() - t0:.0f} s')

    images, depths = np.stack(images), np.stack(depths)
    is_webcam = np.array([f in set(webcam_files) for f in kept])
    # 'files' is the full input list (including skipped ones) so the cache check above matches next time.
    np.savez(LABELS_PATH, images=images, depths=depths, is_webcam=is_webcam, files=np.array(files))
    print(f'Labelled {len(kept)} images ({len(files) - len(kept)} unreadable, skipped) in {time.time() - t0:.0f} s')
    del teacher
    torch.cuda.empty_cache()

print('images:', images.shape, images.dtype, '| depths:', depths.shape, depths.dtype)

Quick check that the labels look right: red = near, blue = far.

In [ ]:
rng = np.random.default_rng(0)
show = rng.choice(len(images), 4, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for col, i in enumerate(show):
    axes[0, col].imshow(images[i])
    axes[1, col].imshow(depths[i].astype(np.float32), cmap='Spectral_r', vmin=0, vmax=1)
    axes[0, col].set_title('webcam' if is_webcam[i] else 'pet photo')
for ax in axes.flat:
    ax.axis('off')
axes[1, 0].set_title('teacher depth')
plt.tight_layout()
plt.show()

## Step 4: Train the student

All images and labels fit on the GPU (under 1 GB), so batches are sliced straight from memory instead of going through a `DataLoader`. This also avoids the Windows worker-process problem from the pets notebook.

- **Loss:** L1 between the student's output and the teacher's depth map.
- **Augmentation:** random left-right flips (image and depth flipped together).
- **Validation:** 10% of the images are held out; the checkpoint with the lowest validation loss is saved to `checkpoints/student_unet.pth` (git-ignored).
- **Mixed precision** (`autocast`) makes training roughly twice as fast on the GPU with no change to the result quality.

In [ ]:
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
VAL_FRACTION = 0.1

X = torch.from_numpy(images).to(DEVICE)                 # N x H x W x 3 uint8
Y = torch.from_numpy(depths).to(DEVICE).unsqueeze(1)    # N x 1 x H x W float16

perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0)).to(DEVICE)
n_val = int(len(X) * VAL_FRACTION)
val_idx, train_idx = perm[:n_val], perm[n_val:]

student = UNet(3, 1).to(DEVICE)
optimizer = torch.optim.Adam(student.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
use_amp = DEVICE == 'cuda'


def predict(net, idx):
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_amp):
        return net(to_input(X[idx], DEVICE)).float()


history = {'train': [], 'val': []}
best_val = float('inf')
t0 = time.time()
for epoch in range(EPOCHS):
    student.train()
    shuffled = train_idx[torch.randperm(len(train_idx), device=DEVICE)]
    total = 0.0
    for b in range(0, len(shuffled), BATCH_SIZE):
        idx = shuffled[b:b + BATCH_SIZE]
        x, y = to_input(X[idx], DEVICE), Y[idx].float()
        flip = torch.rand(len(idx), device=DEVICE) < 0.5
        x[flip], y[flip] = x[flip].flip(-1), y[flip].flip(-1)
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_amp):
            loss = F.l1_loss(student(x).float(), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(idx)
    scheduler.step()

    student.eval()
    with torch.no_grad():
        val = sum(F.l1_loss(predict(student, val_idx[b:b + BATCH_SIZE]), Y[val_idx[b:b + BATCH_SIZE]].float(),
                            reduction='sum').item() for b in range(0, n_val, BATCH_SIZE)) / (n_val * IMG_H * IMG_W)
    history['train'].append(total / len(train_idx))
    history['val'].append(val)
    saved = ''
    if val < best_val:
        best_val = val
        torch.save(student.state_dict(), STUDENT_PATH)
        saved = '  <- saved'
    print(f'Epoch {epoch + 1:2d}/{EPOCHS} | train L1 {history["train"][-1]:.4f} | val L1 {val:.4f} | {time.time() - t0:.0f} s{saved}')

plt.plot(range(1, EPOCHS + 1), history['train'], 'b-o', label='train')
plt.plot(range(1, EPOCHS + 1), history['val'], 'r--s', label='validation')
plt.xlabel('epoch')
plt.ylabel('L1 loss (depth is 0-1)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()

## Step 5: How close is the student to the teacher?

On the held-out images:

- **Val L1:** average per-pixel error, on a 0-1 depth scale. 0.05 means the student is off by 5% of the image's depth range on average.
- **NEAR/FAR cell agreement:** split each depth map into the Level 1 6x8 grid and check whether the student picks the same nearest and farthest cell as the teacher. This is closer to what the drone actually needs than per-pixel error.
- **Speed:** time per webcam-sized frame for both models, including resizing.

In [ ]:
student.load_state_dict(torch.load(STUDENT_PATH, map_location=DEVICE))
student.eval()
with torch.no_grad():
    preds = torch.cat([predict(student, val_idx[b:b + BATCH_SIZE]) for b in range(0, n_val, BATCH_SIZE)])
targets = Y[val_idx].float()

# Per-cell mean closeness on the 6x8 grid (IMG_H=120, IMG_W=160 divide evenly into 6x8 cells of 20x20).
cells = lambda d: F.adaptive_avg_pool2d(d, (6, 8)).flatten(1)
p_cells, t_cells = cells(preds), cells(targets)
near_match = (p_cells.argmax(1) == t_cells.argmax(1)).float().mean().item()
far_match = (p_cells.argmin(1) == t_cells.argmin(1)).float().mean().item()
print(f'Val L1: {F.l1_loss(preds, targets).item():.4f}')
print(f'Same NEAR cell as teacher: {near_match:.0%}   Same FAR cell as teacher: {far_match:.0%}   (chance: 2%)')

# Show a few held-out examples: input, teacher, student, error.
fig, axes = plt.subplots(4, 4, figsize=(14, 10))
for row, k in enumerate(np.linspace(0, n_val - 1, 4).astype(int)):
    t, p = targets[k, 0].cpu().numpy(), preds[k, 0].cpu().numpy()
    axes[row, 0].imshow(images[val_idx[k].item()])
    axes[row, 1].imshow(t, cmap='Spectral_r', vmin=0, vmax=1)
    axes[row, 2].imshow(p, cmap='Spectral_r', vmin=0, vmax=1)
    axes[row, 3].imshow(np.abs(p - t), cmap='magma', vmin=0, vmax=0.5)
for ax, title in zip(axes[0], ['input', 'teacher', 'student', '|student - teacher|']):
    ax.set_title(title)
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from student_unet import StudentDepth

frame = np.random.randint(0, 255, (480, 640, 3), np.uint8)  # webcam-sized dummy frame
teacher = DepthAnythingV2(encoder='vits', features=64, out_channels=[48, 96, 192, 384])
teacher.load_state_dict(torch.load('checkpoints/depth_anything_v2_vits.pth', map_location='cpu'))
models = {'teacher (Depth Anything V2, 308)': teacher.to(DEVICE).eval(), 'student (U-Net, 160x120)': StudentDepth(STUDENT_PATH, DEVICE)}

for name, m in models.items():
    params = sum(p.numel() for p in (m.net if isinstance(m, StudentDepth) else m).parameters())
    for _ in range(5):  # warm-up
        m.infer_image(frame, 308)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(50):
        m.infer_image(frame, 308)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    ms = (time.time() - t0) / 50 * 1000
    print(f'{name:34s} {params / 1e6:5.1f}M params  {ms:6.1f} ms/frame  ({1000 / ms:.0f} FPS max)')
del teacher, models

## Step 6: Run the student on the webcam

Open `Depth-Anything-V2-Webcam-Test.ipynb`, set `MODEL = 'student'` at the top of the Step 3 cell, then re-run Steps 2, 3 and 4 there. The FPS label shows which model is running. Set it back to `'teacher'` to compare.

If the student looks poor on the webcam even though Step 5 looks fine, that is the gap between pet photos and your room: capture webcam frames in Step 2 here, then re-run Steps 3 to 5.